<a href="https://colab.research.google.com/github/suasgn/s2/blob/main/base-model-indobartv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install deps

In [1]:
! pip install datasets evaluate transformers[torch] rouge-score nltk nusacrowd jsonlines indobenchmark-toolkit sentencepiece
# fix from https://github.com/indobenchmark/indobenchmark-toolkit/issues/3
! pip install accelerate==0.25.0
! pip install transformers==4.33.1
! pip install tokenizers==0.13.3
! pip install -U datasets
# ! pip install --no-cache-dir "git+https://github.com/suasgn/Top2Vec.git"

! apt install git-lfs

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.2/384.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.9 MB/s eta 0:00:00
  

## Set environment

In [2]:
from huggingface_hub import notebook_login, login
from google.colab import userdata
import os

login(userdata.get("HF_TOKEN"))
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"


In [3]:
model_checkpoint = "indobenchmark/indobart-v2"

## Croscek dependensi

In [4]:
import accelerate
print(accelerate.__version__)

0.25.0


/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [5]:
import os, transformers
root = os.path.dirname(transformers.__file__)
print("transformers version:", transformers.__version__)
print("transformers path   :", root)
print("DETA file exists    :", os.path.exists(os.path.join(root, "models", "deta", "configuration_deta.py")))

transformers version: 4.33.1
transformers path   : /usr/local/lib/python3.11/dist-packages/transformers
DETA file exists    : True


In [6]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## Kostum Tokenizer [inlined]
adapted supaya bisa jalan di google colab

In [40]:
import os
from shutil import copyfile
from typing import TYPE_CHECKING, Any, Dict, List, NamedTuple, Optional, Sequence, Tuple, Union
from transformers import PreTrainedTokenizer, BatchEncoding

from collections.abc import Mapping
from transformers.tokenization_utils import PaddingStrategy, TensorType
from transformers import (
    is_tf_available,
    is_torch_available
)
import sentencepiece as spm

from transformers.utils import logging
from transformers.utils.generic import _is_tensorflow, _is_torch

logger = logging.get_logger(__name__)

VOCAB_FILES_NAMES = {"vocab_file": "sentencepiece.bpe.model"}

PRETRAINED_VOCAB_FILES_MAP = {
    "vocab_file": {
        "indobenchmark/indobart": "https://huggingface.co/indobenchmark/indobart/resolve/main/sentencepiece.bpe.model",
        "indobenchmark/indogpt": "https://huggingface.co/indobenchmark/indogpt/resolve/main/sentencepiece.bpe.model",
        "indobenchmark/indobart-v2": "https://huggingface.co/indobenchmark/indobart-v2/resolve/main/sentencepiece.bpe.model"
    }
}

PRETRAINED_POSITIONAL_EMBEDDINGS_SIZES = {
    "indobenchmark/indobart": 768,
    "indobenchmark/indogpt": 768,
    "indobenchmark/indobart-v2": 768
}

SHARED_MODEL_IDENTIFIERS = [
    # Load with
    "indobenchmark/indobart",
    "indobenchmark/indogpt",
    "indobenchmark/indobart-v2"
]

SPIECE_UNDERLINE = "▁"

# Define type aliases and NamedTuples
TextInput = str
PreTokenizedInput = List[str]
EncodedInput = List[int]
TextInputPair = Tuple[str, str]
PreTokenizedInputPair = Tuple[List[str], List[str]]
EncodedInputPair = Tuple[List[int], List[int]]

def to_py_obj(obj):
    """
    Convert a TensorFlow tensor, PyTorch tensor, Numpy array or python list to a python list.
    """
    if isinstance(obj, (dict, UserDict)):
        return {k: to_py_obj(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [to_py_obj(o) for o in obj]
    elif is_tf_tensor(obj):
        return obj.numpy().tolist()
    elif is_torch_tensor(obj):
        return obj.detach().cpu().tolist()
    elif is_jax_tensor(obj):
        return np.asarray(obj).tolist()
    elif isinstance(obj, (np.ndarray, np.number)):  # tolist also works on 0d np arrays
        return obj.tolist()
    else:
        return obj

class IndoNLGTokenizer(PreTrainedTokenizer):
    vocab_files_names = VOCAB_FILES_NAMES
    pretrained_vocab_files_map = PRETRAINED_VOCAB_FILES_MAP
    max_model_input_sizes = PRETRAINED_POSITIONAL_EMBEDDINGS_SIZES
    model_input_names=['input_ids', 'attention_mask', 'decoder_input_ids', 'decoder_attention_mask', 'labels']
    input_error_message = "text input must of type `str` (single example), `List[str]` (batch of examples)."

    def __init__(
        self,
        vocab_file,
        decode_special_token=True,
        bos_token="<s>",
        eos_token="</s>",
        sep_token="</s>",
        cls_token="<s>",
        unk_token="<unk>",
        pad_token="<pad>",
        mask_token="<mask>",
        additional_special_tokens=[],
        **kwargs
    ):
        super().__init__(
            vocab_file=vocab_file,
            bos_token=bos_token,
            eos_token=eos_token,
            unk_token=unk_token,
            sep_token=sep_token,
            cls_token=cls_token,
            pad_token=pad_token,
            mask_token=mask_token,
            additional_special_tokens=additional_special_tokens,
            **kwargs,
        )
        self.sp_model = spm.SentencePieceProcessor()
        self.sp_model.Load(str(vocab_file))
        self.vocab_file = vocab_file
        self.decode_special_token = decode_special_token
        self.model_max_length = 1024

        # HACK: These tokens were added by fairseq but don't seem to be actually used when duplicated in the actual
        # sentencepiece vocabulary (this is the case for <s> and </s>
        self.special_tokens_to_ids = {
            "[javanese]": 40000,
            "[sundanese]": 40001,
            "[indonesian]": 40002,
            "<mask>": 40003
        }
        self.special_ids_to_tokens = {v: k for k, v in self.special_tokens_to_ids.items()}

        # Store Language token ID
        self.javanese_token = '[javanese]'
        self.javanese_token_id = 40000
        self.sundanese_token = '[sundanese]'
        self.sundanese_token_id = 40001
        self.indonesian_token = '[indonesian]'
        self.indonesian_token_id = 40002

        self.special_token_ids = [
            self.bos_token_id, self.eos_token_id, self.sep_token_id, self.cls_token_id,
            self.unk_token_id, self.pad_token_id, self.mask_token_id,
            self.javanese_token_id, self.sundanese_token_id, self.indonesian_token_id
        ]

    def prepare_input_for_generation(self, inputs, model_type='indobart', lang_token='[indonesian]', decoder_inputs=None,
                                             decoder_lang_token='[indonesian]', padding='longest', return_tensors=None):
        """
        Build model inputs for a specified `model_type`. There are two possible `model_type`, i.e., indobart and indogpt.

        When `model_type` is indogpt, `lang_token`, `decoder_inputs`, and `decoder_lang_token` parameters will be ignored
        and the input will be encoded in the gpt2 sequence format as follow:

        - indogpt sequence: ``<s> X``

        When `model_type` is indobart, `inputs` and `lang_token` are used as the sequence and language identifier for the indobart encoder,
        while `decoder_inputs` and `decoder_lang_token` are used as the sequence and language identifier of the decoder

        - indobart encoder sequence: ``X </s> <lang_token_id>``
        - indobart decoder sequences: ``<decoder_lang_token_id> X </s>``

        Args:
            inputs (:obj:`str` or `List[str]`):
                text sequence or list of text sequences to be tokenized.
            model_type (:obj:`str`, defaults to :obj:`indobart`):
                model type to determine the format of the tokenized sequence. Valid values are `indobart` and `indogpt`.
            lang_token (:obj:`str`, defaults to :obj:`[indonesian]`):
                language token to determine the format of the tokenized sequence. Valid values are `[indonesian]`, `[sundanese], and [javanese]`.
            decoder_inputs (:obj:`str` or `List[str]`, `optional`):
                decoder text sequence or list of text sequences to be tokenized.
            decoder_lang_token (:obj:`str`, defaults to :obj:`[indonesian]`):
                decoder language token to determine the format of the tokenized sequence. Valid values are `[indonesian]`, `[sundanese], and [javanese]`.
            padding (:obj:`str`, defaults to :obj:`longest`):
                padding strategy to pad the tokenized sequences. Valid values are `longest`, `max_length`, and `do_not_pad`.
            return_tensors (:obj:`str`, defaults to :obj:`None`):
                Returned tensor type of the tokenized sequence. When set to `None`, the return type will be List[int]. Valid values are `None`, `pt`, and `tf`

        Returns:
            :obj:`Dict`: Dictionary with `input_ids`, `attention_mask`, `decoder_input_ids` (optional), and `decoder_attention_mask` (optional)
        """
        if model_type == 'indogpt':
            # Process indogpt input
            if type(inputs) == str:
                 return self(f'<s> {inputs}', padding=padding, return_tensors=return_tensors)
            elif type(inputs) == list:
                if len(inputs) == 0 or type(inputs[0]) != str:
                    raise ValueError(IndoNLGTokenizer.input_error_message)
                else:
                    return self([f'<s> {input_data}' for input_data in inputs], padding=padding, return_tensors=return_tensors)
            else:
                raise ValueError(IndoNLGTokenizer.input_error_message)
        elif model_type == 'indobart':

            # Process encoder input
            if lang_token not in self.special_tokens_to_ids:
                raise ValueError(f"Unknown lang_token `{lang_token}`, lang_token must be either `[javanese]`, `[sundanese]`, or `[indonesian]`")
            elif type(inputs) == list:
                if len(inputs) == 0 or type(inputs[0]) != str:
                    raise ValueError(IndoNLGTokenizer.input_error_message)
            elif type(inputs) != str:
                raise ValueError(IndoNLGTokenizer.input_error_message)

            lang_id = self.special_tokens_to_ids[lang_token]
            input_batch = self(inputs, return_attention_mask=False)
            if type(inputs) == str:
                input_batch['input_ids'] = [self.bos_token_id] + input_batch['input_ids'] + [self.eos_token_id, lang_id]
            else:
                input_batch['input_ids'] = list(map(lambda input_ids: [self.bos_token_id] + input_ids + [self.eos_token_id, lang_id], input_batch['input_ids']))

            if decoder_inputs is None:
                # Return encoder input
                return self.pad(input_batch, return_tensors=return_tensors)
            else:
                # Process decoder input
                if decoder_lang_token not in self.special_tokens_to_ids:
                    raise ValueError(f"Unknown decoder_lang_token `{decoder_lang_token}`, decoder_lang_token must be either `[javanese]`, `[sundanese]`, or `[indonesian]`")
                elif type(decoder_inputs) == list:
                    if len(decoder_inputs) == 0:
                        raise ValueError(IndoNLGTokenizer.input_error_message)
                    elif type(decoder_inputs[0]) != str:
                        raise ValueError(IndoNLGTokenizer.input_error_message)
                elif type(decoder_inputs) != str:
                    raise ValueError(IndoNLGTokenizer.input_error_message)

                decoder_lang_id = self.special_tokens_to_ids[decoder_lang_token]
                decoder_input_batch = self(decoder_inputs, return_attention_mask=False)

                if type(decoder_inputs) == str:
                    labels = [self.bos_token_id] + decoder_input_batch['input_ids'] + [self.eos_token_id, decoder_lang_id]
                    decoder_input_batch['input_ids'] = [decoder_lang_id, self.bos_token_id] + decoder_input_batch['input_ids'] + [self.eos_token_id]
                else:
                    labels = list(map(lambda input_ids: [self.bos_token_id] + input_ids + [self.eos_token_id, decoder_lang_id], decoder_input_batch['input_ids']))
                    decoder_input_batch['input_ids'] = list(map(lambda input_ids: [decoder_lang_id, self.bos_token_id] + input_ids + [self.eos_token_id], decoder_input_batch['input_ids']))

                # Padding
                input_batch = self.pad(input_batch, return_tensors=return_tensors)
                decoder_input_batch = self.pad(decoder_input_batch, return_tensors=return_tensors)
                labels = self.pad({'input_ids': labels}, return_tensors=return_tensors)['input_ids']
                if not isinstance(labels, (list, tuple)):
                    labels[labels == self.pad_token_id] = -100
                else:
                    labels = list(map(lambda x: -100 if x == self.pad_token_id else x, labels))

                # Store into a single dict
                input_batch['decoder_input_ids'] = decoder_input_batch['input_ids']
                input_batch['decoder_attention_mask'] = decoder_input_batch['attention_mask']
                input_batch['labels'] = labels

                return input_batch

    def __len__(self):
        return max(self.special_ids_to_tokens) + 1

    def get_special_tokens_mask(
        self, token_ids_0: List[int], token_ids_1: Optional[List[int]] = None, already_has_special_tokens: bool = False
    ) -> List[int]:
        """
        Retrieve sequence ids from a token list that has no special tokens added. This method is called when adding
        special tokens using the tokenizer ``prepare_for_model`` method.

        Args:
            token_ids_0 (:obj:`List[int]`):
                List of IDs.
            token_ids_1 (:obj:`List[int]`, `optional`):
                Optional second list of IDs for sequence pairs.
            already_has_special_tokens (:obj:`bool`, `optional`, defaults to :obj:`False`):
                Whether or not the token list is already formatted with special tokens for the model.

        Returns:
            :obj:`List[int]`: A list of integers in the range [0, 1]: 1 for a special token, 0 for a sequence token.
        """
        if already_has_special_tokens:
            return super().get_special_tokens_mask(
                token_ids_0=token_ids_0, token_ids_1=token_ids_1, already_has_special_tokens=True
            )

        if token_ids_1 is None:
            return [1] + ([0] * len(token_ids_0)) + [1]
        return [1] + ([0] * len(token_ids_0)) + [1, 1] + ([0] * len(token_ids_1)) + [1]

    @property
    def vocab_size(self):
        return 4 + len(self.sp_model)

    def get_vocab(self):
        vocab = {self.convert_ids_to_tokens(i): i for i in range(self.vocab_size)}
        vocab.update(self.added_tokens_encoder)
        return vocab

    def _tokenize(self, text: str) -> List[str]:
        return self.sp_model.encode(text.lower(), out_type=str)

    def convert_ids_to_tokens(
        self, ids: Union[int, List[int]], skip_special_tokens: bool = False
    ) -> Union[str, List[str]]:
        """
        Converts a single index or a sequence of indices in a token or a sequence of tokens, using the vocabulary and
        added tokens.
        Args:
            ids (`int` or `List[int]`):
                The token id (or token ids) to convert to tokens.
            skip_special_tokens (`bool`, *optional*, defaults to `False`):
                Whether or not to remove special tokens in the decoding.
        Returns:
            `str` or `List[str]`: The decoded token(s).
        """
        if isinstance(ids, int):
            if ids not in self.added_tokens_decoder or ids in self.special_tokens_to_ids:
                return self._convert_id_to_token(ids, skip_special_tokens=skip_special_tokens)
            else:
                return self.added_tokens_decoder[ids]
        tokens = []
        for index in ids:
            index = int(index)
            if skip_special_tokens and index in (self.all_special_ids + list(self.special_tokens_to_ids.values())):
                continue
            if index not in self.added_tokens_decoder or index in self.special_tokens_to_ids:
                tokens.append(self._convert_id_to_token(index, skip_special_tokens=skip_special_tokens))
            else:
                tokens.append(self.added_tokens_decoder[index])
        return tokens

    def _convert_token_to_id(self, token):
        """ Converts a token (str) in an id using the vocab. """
        if token in self.special_tokens_to_ids:
            return self.special_tokens_to_ids[token]
        return self.sp_model.PieceToId(token)

    def _convert_id_to_token(self, index, skip_special_tokens=False):
        """Converts an index (integer) in a token (str) using the vocab."""
        if skip_special_tokens and index in self.special_token_ids:
            return ''

        if index in self.special_ids_to_tokens:
            return self.special_ids_to_tokens[index]

        token = self.sp_model.IdToPiece(index)
        if '<0x' in token:
            char_rep = chr(int(token[1:-1], 0))
            if char_rep.isprintable():
                return char_rep
        return token

    def __getstate__(self):
        state = self.__dict__.copy()
        state["sp_model"] = None
        return state

    def __setstate__(self, d):
        self.__dict__ = d

        # for backward compatibility
        if not hasattr(self, "sp_model_kwargs"):
            self.sp_model_kwargs = {}

        self.sp_model = spm.SentencePieceProcessor(**self.sp_model_kwargs)
        self.sp_model.Load(self.vocab_file)

    def decode(self, inputs, skip_special_tokens=False):
        outputs = super().decode(inputs, skip_special_tokens=skip_special_tokens)
        return outputs.replace(' ','').replace('▁', ' ')

    def _pad_decoder(
        self,
        encoded_inputs: Union[Dict[str, EncodedInput], BatchEncoding],
        max_length: Optional[int] = None,
        padding_strategy: PaddingStrategy = PaddingStrategy.DO_NOT_PAD,
        pad_to_multiple_of: Optional[int] = None,
        return_attention_mask: Optional[bool] = None,
    ) -> dict:
        """
        Pad encoded inputs (on left/right and up to predefined length or max length in the batch)
        Args:
            encoded_inputs:
                Dictionary of tokenized inputs (`List[int]`) or batch of tokenized inputs (`List[List[int]]`).
            max_length: maximum length of the returned list and optionally padding length (see below).
                Will truncate by taking into account the special tokens.
            padding_strategy: PaddingStrategy to use for padding.
                - PaddingStrategy.LONGEST Pad to the longest sequence in the batch
                - PaddingStrategy.MAX_LENGTH: Pad to the max length (default)
                - PaddingStrategy.DO_NOT_PAD: Do not pad
                The tokenizer padding sides are defined in self.padding_side:
                    - 'left': pads on the left of the sequences
                    - 'right': pads on the right of the sequences
            pad_to_multiple_of: (optional) Integer if set will pad the sequence to a multiple of the provided value.
                This is especially useful to enable the use of Tensor Core on NVIDIA hardware with compute capability
                >= 7.5 (Volta).
            return_attention_mask:
                (optional) Set to False to avoid returning attention mask (default: set to model specifics)
        """
        # Load from model defaults
        if return_attention_mask is None:
            return_attention_mask = "decoder_attention_mask" in self.model_input_names

        required_input = encoded_inputs[self.model_input_names[2]]

        if max_length is not None and pad_to_multiple_of is not None and (max_length % pad_to_multiple_of != 0):
            max_length = ((max_length // pad_to_multiple_of) + 1) * pad_to_multiple_of

        needs_to_be_padded = padding_strategy != PaddingStrategy.DO_NOT_PAD and len(required_input) != max_length

        # Initialize attention mask if not present.
        if return_attention_mask and "decoder_attention_mask" not in encoded_inputs:
            encoded_inputs["decoder_attention_mask"] = [1] * len(required_input)

        if needs_to_be_padded:
            difference = max_length - len(required_input)

            if self.padding_side == "right":
                if return_attention_mask:
                    encoded_inputs["decoder_attention_mask"] = encoded_inputs["decoder_attention_mask"] + [0] * difference
                if "decoder_token_type_ids" in encoded_inputs:
                    encoded_inputs["decoder_token_type_ids"] = (
                        encoded_inputs["decoder_token_type_ids"] + [self.pad_token_type_id] * difference
                    )
                if "decoder_special_tokens_mask" in encoded_inputs:
                    encoded_inputs["decoder_special_tokens_mask"] = encoded_inputs["decoder_special_tokens_mask"] + [1] * difference
                encoded_inputs[self.model_input_names[2]] = required_input + [self.pad_token_id] * difference

                label_input = encoded_inputs[self.model_input_names[4]]
                encoded_inputs[self.model_input_names[4]] = label_input + [-100] * difference
            elif self.padding_side == "left":
                if return_attention_mask:
                    encoded_inputs["decoder_attention_mask"] = [0] * difference + encoded_inputs["decoder_attention_mask"]
                if "decoder_token_type_ids" in encoded_inputs:
                    encoded_inputs["decoder_token_type_ids"] = [self.pad_token_type_id] * difference + encoded_inputs[
                        "decoder_token_type_ids"
                    ]
                if "decoder_special_tokens_mask" in encoded_inputs:
                    encoded_inputs["decoder_special_tokens_mask"] = [1] * difference + encoded_inputs["decoder_special_tokens_mask"]
                encoded_inputs[self.model_input_names[2]] = [self.pad_token_id] * difference + required_input

                label_input = encoded_inputs[self.model_input_names[4]]
                encoded_inputs[self.model_input_names[4]] = label_input + [-100] * difference
            else:
                raise ValueError("Invalid padding strategy:" + str(self.padding_side))

        return encoded_inputs

    def decode(self, inputs, skip_special_tokens=False, clean_up_tokenization_spaces: bool = True):
        outputs = super().decode(inputs, skip_special_tokens=skip_special_tokens, clean_up_tokenization_spaces=clean_up_tokenization_spaces)
        return outputs.replace(' ','').replace('▁', ' ')

    def pad(
        self,
        encoded_inputs: Union[
            BatchEncoding,
            List[BatchEncoding],
            Dict[str, EncodedInput],
            Dict[str, List[EncodedInput]],
            List[Dict[str, EncodedInput]],
        ],
        padding: Union[bool, str, PaddingStrategy] = True,
        max_length: Optional[int] = None,
        pad_to_multiple_of: Optional[int] = None,
        return_attention_mask: Optional[bool] = None,
        return_tensors: Optional[Union[str, TensorType]] = None,
        verbose: bool = True,
    ) -> BatchEncoding:
        """
        Pad a single encoded input or a batch of encoded inputs up to predefined length or to the max sequence length
        in the batch.

        Padding side (left/right) padding token ids are defined at the tokenizer level (with `self.padding_side`,
        `self.pad_token_id` and `self.pad_token_type_id`)

        <Tip>

        If the `encoded_inputs` passed are dictionary of numpy arrays, PyTorch tensors or TensorFlow tensors, the
        result will use the same type unless you provide a different tensor type with `return_tensors`. In the case of
        PyTorch tensors, you will lose the specific device of your tensors however.

        </Tip>

        Args:
            encoded_inputs ([`BatchEncoding`], list of [`BatchEncoding`], `Dict[str, List[int]]`, `Dict[str, List[List[int]]` or `List[Dict[str, List[int]]]`):
                Tokenized inputs. Can represent one input ([`BatchEncoding`] or `Dict[str, List[int]]`) or a batch of
                tokenized inputs (list of [`BatchEncoding`], *Dict[str, List[List[int]]]* or *List[Dict[str,
                List[int]]]*) so you can use this method during preprocessing as well as in a PyTorch Dataloader
                collate function.

                Instead of `List[int]` you can have tensors (numpy arrays, PyTorch tensors or TensorFlow tensors), see
                the note above for the return type.
            padding (`bool`, `str` or [`~utils.PaddingStrategy`], *optional*, defaults to `True`):
                 Select a strategy to pad the returned sequences (according to the model's padding side and padding
                 index) among:

                - `True` or `'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
                  sequence if provided).
                - `'max_length'`: Pad to a maximum length specified with the argument `max_length` or to the maximum
                  acceptable input length for the model if that argument is not provided.
                - `False` or `'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of different
                  lengths).
            max_length (`int`, *optional*):
                Maximum length of the returned list and optionally padding length (see above).
            pad_to_multiple_of (`int`, *optional*):
                If set will pad the sequence to a multiple of the provided value.

                This is especially useful to enable the use of Tensor Cores on NVIDIA hardware with compute capability
                >= 7.5 (Volta).
            return_attention_mask (`bool`, *optional*):
                Whether to return the attention mask. If left to the default, will return the attention mask according
                to the specific tokenizer's default, defined by the `return_outputs` attribute.

                [What are attention masks?](../glossary#attention-mask)
            return_tensors (`str` or [`~utils.TensorType`], *optional*):
                If set, will return tensors instead of list of python integers. Acceptable values are:

                - `'tf'`: Return TensorFlow `tf.constant` objects.
                - `'pt'`: Return PyTorch `torch.Tensor` objects.
                - `'np'`: Return Numpy `np.ndarray` objects.
            verbose (`bool`, *optional*, defaults to `True`):
                Whether or not to print more information and warnings.
        """
        # If we have a list of dicts, let's convert it in a dict of lists
        # We do this to allow using this method as a collate_fn function in PyTorch Dataloader
        if isinstance(encoded_inputs, (list, tuple)) and isinstance(encoded_inputs[0], Mapping):
            encoded_inputs = {key: [example[key] for example in encoded_inputs] for key in encoded_inputs[0].keys()}

        # The model's main input name, usually `input_ids`, has be passed for padding
        if self.model_input_names[0] not in encoded_inputs:
            raise ValueError(
                "You should supply an encoding or a list of encodings to this method "
                f"that includes {self.model_input_names[0]}, but you provided {list(encoded_inputs.keys())}"
            )

        required_input = encoded_inputs[self.model_input_names[0]]

        if not required_input:
            if return_attention_mask:
                encoded_inputs["attention_mask"] = []
            return encoded_inputs

        # If we have PyTorch/TF/NumPy tensors/arrays as inputs, we cast them as python objects
        # and rebuild them afterwards if no return_tensors is specified
        # Note that we lose the specific device the tensor may be on for PyTorch

        first_element = required_input[0]
        if isinstance(first_element, (list, tuple)):
            # first_element might be an empty list/tuple in some edge cases so we grab the first non empty element.
            for item in required_input:
                if len(item) != 0:
                    first_element = item[0]
                    break
        # At this state, if `first_element` is still a list/tuple, it's an empty one so there is nothing to do.
        if not isinstance(first_element, (int, list, tuple)):
            if is_tf_available() and _is_tensorflow(first_element):
                return_tensors = "tf" if return_tensors is None else return_tensors
            elif is_torch_available() and _is_torch(first_element):
                return_tensors = "pt" if return_tensors is None else return_tensors
            elif isinstance(first_element, np.ndarray):
                return_tensors = "np" if return_tensors is None else return_tensors
            else:
                raise ValueError(
                    f"type of {first_element} unknown: {type(first_element)}. "
                    f"Should be one of a python, numpy, pytorch or tensorflow object."
                )

            for key, value in encoded_inputs.items():
                encoded_inputs[key] = to_py_obj(value)

        # Convert padding_strategy in PaddingStrategy
        padding_strategy, _, max_length, _ = self._get_padding_truncation_strategies(
            padding=padding, max_length=max_length, verbose=verbose
        )

        required_input = encoded_inputs[self.model_input_names[0]]
        if required_input and not isinstance(required_input[0], (list, tuple)):
            encoded_inputs = self._pad(
                encoded_inputs,
                max_length=max_length,
                padding_strategy=padding_strategy,
                pad_to_multiple_of=pad_to_multiple_of,
                return_attention_mask=return_attention_mask,
            )
            return BatchEncoding(encoded_inputs, tensor_type=return_tensors)

        batch_size = len(required_input)
        assert all(
            len(v) == batch_size for v in encoded_inputs.values()
        ), "Some items in the output dictionary have a different batch size than others."

        if padding_strategy == PaddingStrategy.LONGEST:
            max_length = max(len(inputs) for inputs in required_input)
            padding_strategy = PaddingStrategy.MAX_LENGTH

        batch_outputs = {}
        for i in range(batch_size):
            inputs = dict((k, v[i]) for k, v in encoded_inputs.items())
            outputs = self._pad(
                inputs,
                max_length=max_length,
                padding_strategy=padding_strategy,
                pad_to_multiple_of=pad_to_multiple_of,
                return_attention_mask=return_attention_mask,
            )

            # Handle decoder_input_ids
            if self.model_input_names[2] in outputs:
                max_decoder_length = max(len(inputs) for inputs in encoded_inputs[self.model_input_names[2]])
                outputs = self._pad_decoder(
                    outputs,
                    max_length=max_decoder_length,
                    padding_strategy=padding_strategy,
                    pad_to_multiple_of=pad_to_multiple_of,
                    return_attention_mask=return_attention_mask,
                )

            for key, value in outputs.items():
                if key not in batch_outputs:
                    batch_outputs[key] = []
                batch_outputs[key].append(value)

        return BatchEncoding(batch_outputs, tensor_type=return_tensors)

    def save_vocabulary(self, save_directory: str, filename_prefix: Optional[str] = None) -> Tuple[str]:
        if not os.path.isdir(save_directory):
            logger.error(f"Vocabulary path ({save_directory}) should be a directory")
            return
        out_vocab_file = os.path.join(save_directory, (filename_prefix + "-" if filename_prefix else "") + VOCAB_FILES_NAMES["vocab_file"])

        if os.path.abspath(self.vocab_file) != os.path.abspath(out_vocab_file):
            copyfile(self.vocab_file, out_vocab_file)
        return (out_vocab_file,)

## Loading datasets

In [8]:
import datasets
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=5):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [9]:
from datasets import load_dataset
import json

raw_datasets = load_dataset("joshuasiagian/indosum")

# rename kolom paragraph menjadi document
raw_datasets = raw_datasets.rename_column("paragraph", "document")

README.md: 0.00B [00:00, ?B/s]

data/train.00.jsonl:   0%|          | 0.00/36.3M [00:00<?, ?B/s]

data/dev.00.jsonl:   0%|          | 0.00/1.92M [00:00<?, ?B/s]

data/test.00.jsonl:   0%|          | 0.00/9.52M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14262 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3762 [00:00<?, ? examples/s]

### lihat dataset

In [10]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['document', 'summary'],
        num_rows: 14262
    })
    validation: Dataset({
        features: ['document', 'summary'],
        num_rows: 750
    })
    test: Dataset({
        features: ['document', 'summary'],
        num_rows: 3762
    })
})

In [11]:
raw_datasets["train"][0]

{'document': 'Jakarta, CNN Indonesia - - Dokter Ryan Thamrin, yang terkenal lewat acara Dokter Oz Indonesia, meninggal dunia pada Jumat (4 / 8) dini hari. Dokter Lula Kamal yang merupakan selebriti sekaligus rekan kerja Ryan menyebut kawannya itu sudah sakit sejak setahun yang lalu. Lula menuturkan, sakit itu membuat Ryan mesti vakum dari semua kegiatannya, termasuk menjadi pembawa acara Dokter Oz Indonesia. Kondisi itu membuat Ryan harus kembali ke kampung halamannya di Pekanbaru, Riau untuk menjalani istirahat. " Setahu saya dia orangnya sehat, tapi tahun lalu saya dengar dia sakit. (Karena) sakitnya, ia langsung pulang ke Pekanbaru, jadi kami yang mau jenguk juga susah. Barangkali mau istirahat, ya betul juga, kalau di Jakarta susah isirahatnya, " kata Lula kepada CNNIndonesia.com, Jumat (4 / 8). Lula yang mengenal Ryan sejak sebelum aktif berkarier di televisi mengaku belum sempat membesuk Ryan lantaran lokasi yang jauh. Dia juga tak tahu penyakit apa yang diderita Ryan. " Itu saya

In [12]:
show_random_elements(raw_datasets["train"])

,document,summary
0,"LONDON, JUARA.net - Huddersfield Town menjadi tim ketiga yang memastikan promosi ke Premier League musim depan setelah mengalahkan Reading lewat drama adu penalti pada partai final play-off Championship di Stadion Wembley, Senin (29 / 5 / 2017) waktu setempat. Reading mencapai final play-off setelah mengalahkan Fulham dengan agregat 2 - 1 pada semifinal. Sementara itu, Huddersfield menyingkirkan Sheffield Wednesday lewat drama adu penalti setelah agregat imbang 1 - 1. Dalam pertandingan ini kedua tim terlihat memiliki kekuatan berimbang. Laga pun cenderung berjalan alot pada babak pertama. Huddersfield dan Reading hanya mampu menciptakan satu peluang bagus pada 45 menit pertama. Peluang terbaik Reading tercipta pada menit ke - 35. Namun, sepakan John Swift tidak maksimal. Sementara itu, tendangan kaki kiri pemain Huddersfield, Isaiah Brown, pada menit ke - 37 berhasil diantisipasi kiper Reading, Ali Al Habsi. Pada awal babak kedua Huddersfield langsung mendapatkan peluang bagus lewat Chris Lowe. Namun, upaya tersebut kembali digagalkan Al Habsi. Setelah peluang tersebut pertandingan kembali berlangsung alot lantaran kedua tim sama-sama kesulitan memecahkan kebuntuan. Tetap tidak ada gol hingga paruh kedua berakhir, laga ini pun dilanjutkan ke babak perpanjangan waktu selama 2x15 menit. Skor 0 - 0 masih bertahan hingga 120 menit bergulir. Alhasil, pemenang laga harus ditentukan lewat adu tendangan penalti. Dalam drama ini penendang kedua Huddersfield, Michael Hefele, gagal mencetak gol sehingga Reading dalam posisi unggul hingga 3 - 2. Namun, eksekutor keempat dan kelima Reading, yakni Liam Moore dan Jordan Obita gagal cetak gol. Dua gol dari Aaron Mooy dan Christopher Schindler pun memastikan kemenangan Huddersfield dengan skor 4 - 3 dalam adu penalti. Kemenangan ini membuat Huddersfield menjadi tim terakhir yang berhak berlaga di Premier League pada musim 2017 - 2018. Dua tim lain yang sudah lebih dulu memastikan promosi ke Premier League, yakni Newcastle United dan Brighton & Hove Albion. Newcastle kembali ke kompetisi paling bergengsi di Inggris seusai finis sebagai juara divisi Championship. Sedangkan, Brighton & Hove Albion sebagai runner-up.","Huddersfield Town menjadi tim ketiga yang memastikan promosi ke Premier League musim depan setelah mengalahkan Reading lewat drama adu penalti pada partai final play-off Championship di Stadion Wembley, Senin (29 / 5 / 2017) waktu setempat. Reading mencapai final play-off setelah mengalahkan Fulham dengan agregat 2 - 1 pada semifinal. Sementara itu, Huddersfield menyingkirkan Sheffield Wednesday lewat drama adu penalti setelah agregat imbang 1 - 1."
1,"Rimanews - Peneliti menunjukkan mengapa kurang tidur dapat meningkatkan risiko terkena penyakit Alzheimer's. Kurang tidur semalam saja, menurut studi terbaru, dapat menyebabkan peningkatan kadar protein yang disebut beta amyloid. Sementara kadar protein yang disebut tau dapat meningkat jika seseorang tidak mendapatkan waktu tidur yang cukup selama sepekan. Kedua protein itu diyakini dapat menyebabkan penyakit Alzheimer's. "" Kami menunjukkan bahwa kualitas tidur yang buruk terkait dengan tingginya kadar protein pemicu Alzheimer's, "" kata dr. David Holtzman, profesor dan ketua departemen neurologi Fakultas Kedokteran Universitas Washington di St. Louis. Lebih dari 5 juta penduduk Amerika Serikat menderita Alzheimer's, tambah para peneliti dalam sebuah pernyataan, sebagaimana dikutip dari CBS News, Selasa (11 / 07 / 2017). Studi di masa lalu sudah menunjukkan hubungan antara buruknya kualitas tidur dan penyakit yang merusak memori manusia tersebut. Sebagai contoh, orang yang menderita sleep apnea (gangguan serius pada pernapasan yang terjadi saat tidur) menyebabkan gangguan ringan pada kemampuan berpikir dan mengingat seseorang 10 tahun lebih cepat daripada mereka yang tidak mengalami kurang tidur. Kerusakan kognitif ringan bisa menjadi tanda awal penyakit Alzheimer's. Untuk membuktikan bahwa kualitas tidur yang b

In [13]:
show_random_elements(raw_datasets["validation"])

,document,summary
0,"Rimanews - Seorang pria paruh baya, TH (66) tewas bunuh diri dengan cara menyayat pergelangan tangannya dengan pisau daging, usai menganiaya istrinya NL (47), karena cemburu. Warga Perumahan Alam Indah Blok C RT 03 RW 04, Kelurahan Poris, Kecamatan Cipondoh, Kota Tangerang, itu sempat dibawa ke rumah sakit oleh tetangga - tetangganya namun nyawanya tidak tertolong. Kapolsek Cipondoh, Kompol Bayu Suseno mengatakan, peristiwa itu bermula dari perselisihan antara TH dan NL yang berujung pada penganiayaan terhadap NL hingga mengalami luka-luka. “ Percekcokan ini berawal ketika pelaku melihat ponsel milik korban yang ada foto teman SMA -nya. Pelaku cemburu dan mencoba membunuh istrinya, ” katanya seperti dilansir laman Tribratanews, kemarin. Tetangga yang mendengar keributan kemudian menuju tempat tinggal pasangan suami istri ini. Warga juga sempat melerai pertengkaran dan mengamankan NL yang sudah berlumuran darah keluar dari rumah. Sementara pelaku kembali masuk ke rumah dan mengunci pintu. Pukul 01.00 WIB dini hari, polisi mendapat laporan dari warga bahwa TH mengurung diri di dalam rumah. "" Petugas berusaha membuka paksa pintu dengan cara didobrak. Setelah pintu terbuka, didapati pelaku berada di dalam kamar mandi. Namun pintu kamar mandi dikunci, ” kata Kapolsek. Lalu polisi membuka paksa pintu kamar mandi dan didapati sudah banyak darah di lantai. Petugas melihat pergelangan tangan kanan dan tangan kiri pelaku sudah terluka “ Pelaku sempat mendapatkan perawatan medis, tapi sekira pukul 03.00 dini hari pelaku dinyatakan meninggal dunia . "" kata Kapolsek. Kasus ini masih dalam penanganan pihak Polsek Cipondoh. Sementara istri korban NL, kata Bayu belum bisa dimintai keterangan karena masih dalam kondisi syok.","Seorang pria paruh baya, TH (66) tewas bunuh diri dengan cara menyayat pergelangan tangannya dengan pisau daging, usai menganiaya istrinya NL (47), karena cemburu. Warga Perumahan Alam Indah Blok C RT 03 RW 04, Kelurahan Poris, Kecamatan Cipondoh, Kota Tangerang, itu sempat dibawa ke rumah sakit oleh tetangga - tetangganya namun nyawanya tidak tertolong. Peristiwa itu bermula dari perselisihan antara TH dan NL."
1,"Jakarta, CNN Indonesia - - Pemilik jaringan Hotel Sahid Almarhum Sukamdani Sahid Gitosardjono dimakamkan dengan tata cara militer di Pondok Pesantren Modern Sahid, Gunung Menyan, Bogor Jawa Barat. Kamis sore (21 / 12)sekitar pukul empat lebih jenazah bersama rombongan diberangkatkan dengan iring-iringan dari militer dan kepolisian. Ia meninggalkan lima anak yakni Sarwo Budi Wiyanti, Exacty Budiarsi, Nugroho Budisatrio, Hariyadi Budisantoso, dan Sri Bimastuti Handayani. Menurut Putra Sekaligus Presiden Direktur PT Hotel Sahidjaya International Tbk Hariyadi Bidisantoso Sukamdani, almarhum mendapat kesempatan dimakamkan di Taman Makam Pahlawan Kalibata Jakarta Selatan. Hanya saja, almarhum berwasiat untuk dimakamkan di Bogor. "" Iya karena pak Sukamdani mempunyai kesempatan untuk dimakamkan di TMP Kalibata dan pemegang anugerah mahaputera sebelum jadi pengusaha dia adalah tentara, "" ungkap Hariyadi saat ditemui di kawasan Rumah Duka Jalan Imam Bonjol, Jakarta, Kamis (21 / 12). Lebih lanjut Haryadi mengungkapkan, penyebab kematian Sukamdani karena usianya yang sudah terbilang senja menginjak 89 tahun dan ada organ yang malfungsi. Awalnya, Sukamdani dibawa ke Rumah Sakit pada Maret 2017 karena sulit bernafas. "" Waktu beliau colaps pertama karena batuk tak bisa mengeluarkan dahaknya jadi kesedak jadi gagal nafas, "" tuturnya. Semasa hidupnya menurut Hariyadi Sukamdani selalu mengingatkan filosofi hidupnya kepada anaknya, bahwa hidup harus menghidupi. Ia pun mengakui Sukamdani adalah sosok panutan dan unik. "" Pak Sukamdani pribadi yang lengkap semua profesi sudah pernah dilalui beliau adalah sosok yang unik dan bisa jadi panutan untuk semuanya, "" kenangnya. Mengenai berbagai bisnis yang ditinggal oleh Sukamdani, menurut Hariyadi, anak-anak Sukamdani akan bersama-sama melanjutkannya. "" Kami

In [14]:
show_random_elements(raw_datasets["test"])

,document,summary
0,"Jakarta, CNN Indonesia - - Di tangan engineer BMW, kunci mobil tak lagi sekadar alat membuka atau menutup pintu. BMW mendesain Display Key yang menyajikan berbagai informasi mengenai mobil, bahkan untuk memarkirkan mobil. Teknologi telah membuat kunci mobil kini lebih mendekati smartphone ketimbang kunci tradisional. Sebab, di dalamnya telah tertanam berbagai teknologi cerdas. Display Key yang memiliki layar sentuh 2,2 inci dengan resolusi 320x240 piksel itu menampilkan berbagai informasi status mobil. Pengguna juga bisa memilih berbagai fungsi yang ada untuk dikontrol melalui layar sentuh terintegrasi. Informasi yang bisa dilihat antara lain, penunjuk waktu dan status bahan bakar yang tersisa beserta jarak tempuh yang mungkin dicapai dengan sisa bahan bakar itu. Display key juga menampilkan notifikasi untuk service, status terkini sistem central lock, jendela, dan sunroof. Pengguna juga bisa mengatur temperatur di dalam mobil, sebelum masuk ke dalamnya. Daya Display Key dapat di isi dengan menggunakan soket biasa maupun portal USB yang ada di dalam mobil layaknya smartphone Anda.","Kunci mobil tak lagi sekadar alat membuka atau menutup pintu. BMW mendesain Display Key yang menyajikan berbagai informasi mengenai mobil, bahkan untuk memarkirkan mobil. Informasi yang bisa dilihat antara lain, penunjuk waktu dan status bahan bakar yang tersisa beserta jarak tempuh yang mungkin dicapai dengan sisa bahan bakar itu. Teknologi telah membuat kunci tersebut lebih mendekati smartphone ketimbang kunci tradisional."
1,"Jakarta, CNN Indonesia - - Sejumlah koleksi barang langka terkait band legendaris The Beatles dilelang pada Sabtu (18 / 3) lalu di Paris. Barang - barang itu termasuk piringan hitam, foto dan perlengkapan lainnya yang selama ini tak banyak diketahui orang. Menurut laporan Reuters, seorang Beatlemania [pencinta The Beatles] bernama Jacques Volcouve yang melelang barang-barang langka itu. Dia telah mengoleksi sejumlah barang-barang yang berkaitan dengan band itu, sejak 1967. Volcouve mengawali koleksinya lewat album A Hard Day Night. Beberapa dekade setelah itu, koleksinya telah mencapai hampir 15 ribu keping rekaman dan barang-barang lain. "" Mulai dari 1967, saya punya misi untuk memiliki hal yang mustahil bagi diri saya, yakni memiliki sendiri segala sesuatu tentang The Beatles, "" katanya pada Reuters TV. Misi itu kemudian membawa keuntungan baginya. Pada Desember lalu, pria berusia 60 tahun itu mulai memilah - milah koleksi yang dimiliki. Dia memutuskan untuk melelang sejumlah koleksinya itu untuk membiayai masa tuanya. Di antara 332 jenis barang yang dilelang itu, turut termasuk sebuah keping rekaman Tony Sheridan and the Beatles 7: My Bonnie lengkap dengan tanda tangan Paul McCartney dan George Harrison. Barang itu dijual dengan perkiraan harga enam sampai sepuluh ribu euro (Rp 86 - 143 juta). Selain itu, terdapat 11 foto alternatif untuk sampul album Sergeant Pepper Lonely Hearts Club yang diperkirakan bernilai 10 sampai 15 ribu euro (Rp 143 - 215 juta). Volcouve pun melelang sebuah surat ucapan terimakasih dari Ringo Starr serta Harrison Ford berkat artikel yang ditulisnya dalam buku dan sebuah komentar di radio soal The Beatles. Itu dihargainya senilai tiga ribu euro (Rp 43 juta). Barang lainnya yang turut dilelang yakni satu set boneka lengkap dengan alat musik yang mereka mainkan, wig autentik Beatles, serta poster dan album pernikahan Yoko Ono dan John Lennon. Dari semua barang itu, Volcouve bisa dapat ratusan juta rupiah.","Sejumlah koleksi barang langka terkait band legendaris The Beatles dilelang pada Sabtu (18 / 3) lalu di Paris. Barang - barang itu termasuk piringan hitam, foto dan perlengkapan lainnya yang selama ini tak banyak diketahui orang. Seorang Beatlemania ernama Jacques Volcouve yang melelang barang-barang langka itu. Dia telah mengoleksinya sejak 1967. Volcouve mengawali koleksinya lewat album A Hard Day Night."
2,"JUARA.NET - Rencana petarung bebas UFC asal Republik Irlandia,

## Load metrik evaluasi: rouge

In [15]:
from evaluate import load

metric = load("rouge")

/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [16]:
metric

EvaluationModule(name: "rouge", module_type: "metric", features: [{'predictions': Value('string'), 'references': List(Value('string'))}, {'predictions': Value('string'), 'references': Value('string')}], usage: """
Calculates average rouge scores for a list of hypotheses and references
Args:
    predictions: list of predictions to score. Each prediction
        should be a string with tokens separated by spaces.
    references: list of reference for each prediction. Each
        reference should be a string with tokens separated by spaces.
    rouge_types: A list of rouge types to calculate.
        Valid names:
        `"rouge{n}"` (e.g. `"rouge1"`, `"rouge2"`) where: {n} is the n-gram based scoring,
        `"rougeL"`: Longest common subsequence based scoring.
        `"rougeLsum"`: rougeLsum splits text using `"
"`.
        See details in https://github.com/huggingface/datasets/issues/617
    use_stemmer: Bool indicating whether Porter stemmer should be used to strip word suffixes.
 

In [17]:
fake_preds = ["hello there", "general kenobi"]
fake_labels = ["hello there", "general kenobi"]
metric.compute(predictions=fake_preds, references=fake_labels)

{'rouge1': np.float64(1.0),
 'rouge2': np.float64(1.0),
 'rougeL': np.float64(1.0),
 'rougeLsum': np.float64(1.0)}

## Load Tokenizer

In [18]:
tokenizer = IndoNLGTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


sentencepiece.bpe.model:   0%|          | 0.00/932k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/303 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [19]:
special_tokens_dict = {'additional_special_tokens': ['<tag>']}
tokenizer.add_special_tokens(special_tokens_dict)
print(tokenizer)

IndoNLGTokenizer(name_or_path='indobenchmark/indobart-v2', vocab_size=40004, model_max_length=1024, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True), 'additional_special_tokens': ['<tag>']}, clean_up_tokenization_spaces=True)


In [20]:
tokenizer.all_special_tokens

['<s>', '</s>', '<unk>', '<pad>', '<mask>', '<tag>']

In [21]:
print(tokenizer.all_special_ids)

[0, 2, 3, 1, 40003, 40004]


In [22]:
tokenizer("Hello, this one sentence!")

{'input_ids': [11624, 39956, 6473, 5926, 3311, 4597, 39981], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [23]:
tokenizer(["Hello, this one sentence!", "This is another sentence."])

{'input_ids': [[11624, 39956, 6473, 5926, 3311, 4597, 39981], [6473, 701, 32129, 3311, 4597, 39955]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}

In [24]:
print(tokenizer(text_target=["Hello, this one sentence!", "This is another sentence."]))

{'input_ids': [[11624, 39956, 6473, 5926, 3311, 4597, 39981], [6473, 701, 32129, 3311, 4597, 39955]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}


### Tokenized dataset

We can then write the function that will preprocess our samples. We just feed them to the `tokenizer` with the argument `truncation=True`. This will ensure that an input longer that what the model selected can handle will be truncated to the maximum length accepted by the model. The padding will be dealt with later on (in a data collator) so we pad examples to the longest length in the batch and not the whole dataset.

In [42]:
max_input_length = 1024
max_target_length = 128

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["document"],
        max_length=max_input_length,
        truncation=True
    )

    model_inputs["labels"] = tokenizer(
        text_target=examples["summary"],
        max_length=max_target_length,
        truncation=True
    )["input_ids"]

    return model_inputs

In [26]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True, batch_size=20000)

Map:   0%|          | 0/14262 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/3762 [00:00<?, ? examples/s]

In [27]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14262
    })
    validation: Dataset({
        features: ['document', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 750
    })
    test: Dataset({
        features: ['document', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3762
    })
})

In [28]:
tokenized_datasets["train"][0]

{'document': 'Jakarta, CNN Indonesia - - Dokter Ryan Thamrin, yang terkenal lewat acara Dokter Oz Indonesia, meninggal dunia pada Jumat (4 / 8) dini hari. Dokter Lula Kamal yang merupakan selebriti sekaligus rekan kerja Ryan menyebut kawannya itu sudah sakit sejak setahun yang lalu. Lula menuturkan, sakit itu membuat Ryan mesti vakum dari semua kegiatannya, termasuk menjadi pembawa acara Dokter Oz Indonesia. Kondisi itu membuat Ryan harus kembali ke kampung halamannya di Pekanbaru, Riau untuk menjalani istirahat. " Setahu saya dia orangnya sehat, tapi tahun lalu saya dengar dia sakit. (Karena) sakitnya, ia langsung pulang ke Pekanbaru, jadi kami yang mau jenguk juga susah. Barangkali mau istirahat, ya betul juga, kalau di Jakarta susah isirahatnya, " kata Lula kepada CNNIndonesia.com, Jumat (4 / 8). Lula yang mengenal Ryan sejak sebelum aktif berkarier di televisi mengaku belum sempat membesuk Ryan lantaran lokasi yang jauh. Dia juga tak tahu penyakit apa yang diderita Ryan. " Itu saya

## Prepare model

In [30]:
# downgrade the peft, as the new version trying to import Cache from transformers
!pip install peft==0.5.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 8.6 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.16.0
    Uninstalling peft-0.16.0:
      Successfully uninstalled peft-0.16.0


In [31]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [32]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# resize karena kita menambahkan `<tag>` sebelumnya di tokenizer
model.resize_token_embeddings(len(tokenizer) + len(tokenizer.added_tokens_encoder))

print(model)

pytorch_model.bin:   0%|          | 0.00/526M [00:00<?, ?B/s]

You are resizing the embedding layer without providing a `pad_to_multiple_of` parameter. This means that the new embedding dimension will be 40005. This might induce some performance reduction as *Tensor Cores* will not be available. For more details about this, or help on choosing the correct value for resizing, refer to this guide: https://docs.nvidia.com/deeplearning/performance/dl-performance-matrix-multiplication/index.html#requirements-tc


MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): Embedding(40005, 768)
    (encoder): MBartEncoder(
      (embed_tokens): Embedding(40005, 768)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affin

In [33]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

## Training & Evaluation

## Metrics

In [34]:
import nltk
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Rouge expects a newline after each sentence
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    ## below are commented out because it take resource and loading time
    # print("decoded_preds")
    # print(decoded_preds)
    # print("decoded_labels")
    # print(decoded_labels)

    # Note that other metrics may not have a `use_aggregator` parameter
    # and thus will return a list, computing a metric for each sentence.
    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True, use_aggregator=True)
    # Extract a few results
    result = {key: value * 100 for key, value in result.items()}

    # Add mean generated length
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

### Training

In [35]:
model_name = model_checkpoint.split("/")[-1]

In [36]:
# hyperparameter - finetuning
batch_size = 8
learning_rate = 3.75e-5
weight_decay = 0.01
num_train_epochs = 3

In [37]:
args = Seq2SeqTrainingArguments(
    f"{model_name}-finetuned-indosum",
    evaluation_strategy="epoch",
    save_strategy="epoch",  # penting
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=weight_decay,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    generation_num_beams=4,  # tambahan
    fp16=True,
    push_to_hub=False,
    save_total_limit=2,  # aman
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    generation_max_length=85,
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [38]:
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [43]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,0.354900,0.540696,68.347300,60.126100,64.674400,67.506500,84.972000


NotImplementedError: 

### Evaluasi (3 split)

In [ ]:
train_results = trainer.predict(tokenized_datasets["train"])
train_metrics = train_results.metrics
print(train_metrics)

In [ ]:
val_results   = trainer.predict(tokenized_datasets["validation"])
val_metrics   = val_results.metrics
print(train_metrics)

In [ ]:
test_results  = trainer.predict(tokenized_datasets["test"])
test_metrics  = test_results.metrics
print(test_metrics)

## List dependency

In [ ]:
!pip list